In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer_probe_epoch50_patience10 import SwinTransformer3D, VSTFCCCLinearProbe
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:2' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 02:22:49,441]::2853905153::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:2


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:
backbone_checkpoint_path = './save_swintransformer_folder/model_best_multiseed_direct_mse.pth'
probe_best_model_path = './save_swintransformer_folder/vst_fccc_linear_probe_best.pth'
os.makedirs(os.path.dirname(probe_best_model_path), exist_ok=True)
print(1)


1


In [6]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 50
print(1)

1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0], fullshot=train_set_data[1], tagdata=train_set_data[2], transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0], fullshot=validation_set_data[1], tagdata=validation_set_data[2], transform=transform)
print(1)


1


In [11]:
criterion = torch.nn.MSELoss().to(device)
print(1)


1


In [12]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [13]:
SEEDS = [42]
patience = 10

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def load_backbone_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    return model

seed_results = []
global_best_mean_mse = float('inf')
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(2):

    for seed_idx, seed in enumerate(SEEDS, start=1):

        logger.info(
            '================ Probe Seed {}/{}: {} ================'.format(
                seed_idx, len(SEEDS), seed
            )
        )

        set_seed(seed)

        train_generator = torch.Generator()
        train_generator.manual_seed(seed)

        val_generator = torch.Generator()
        val_generator.manual_seed(seed + 10000)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=train_generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=val_generator
        )

        # ----------------------------------------------------
        # Frozen trained backbone + freshly initialized probes
        # ----------------------------------------------------
        backbone = SwinTransformer3D()
        backbone = load_backbone_checkpoint(
            backbone,
            backbone_checkpoint_path
        )

        model = VSTFCCCLinearProbe(
            backbone=backbone,
            num_outputs=5
        ).to(device)

        optimizer = torch.optim.AdamW(
            model.probe_heads.parameters(),
            lr=lr
        )

        probe_names = list(model.probe_heads.keys())

        seed_best = {
            name: {
                'mse': float('inf'),
                'epoch': 0
            }
            for name in probe_names
        }

        seed_best_mean_mse = float('inf')
        no_improve_count = 0

        # ----------------------------------------------------
        # Sanity checks
        # ----------------------------------------------------
        logger.info(
            'Backbone trainable params: {}'.format(
                sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
            )
        )
        logger.info(
            'Probe trainable params: {}'.format(
                sum(p.numel() for p in model.probe_heads.parameters() if p.requires_grad)
            )
        )
        logger.info('Probe dimensions: {}'.format(model.probe_dims))

        for epoch in range(epochs):

            # ==================================================
            # Train probes only
            # ==================================================
            model.train()

            train_sq_sum = {name: 0.0 for name in probe_names}
            train_count = {name: 0 for name in probe_names}

            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc='Probe Seed {} Train Epoch {}'.format(seed, epoch + 1)
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1).squeeze()
                if big_five_data.ndim == 1:
                    big_five_data = big_five_data.unsqueeze(0)

                audiodata = audiodata.to(device)
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                outputs = model(
                    fullshot,
                    audiodata
                )

                losses = {
                    name: criterion(outputs[name], big_five_data)
                    for name in probe_names
                }

                loss = torch.stack(
                    [losses[name] for name in probe_names]
                ).mean()

                loss.backward()
                optimizer.step()

                with torch.no_grad():
                    for name in probe_names:
                        train_sq_sum[name] += (
                            outputs[name] - big_five_data
                        ).square().sum().item()
                        train_count[name] += big_five_data.numel()

            train_mse = {
                name: train_sq_sum[name] / train_count[name]
                for name in probe_names
            }

            # ==================================================
            # Validation
            # ==================================================
            model.eval()

            val_sq_sum = {name: 0.0 for name in probe_names}
            val_trait_sq_sum = {name: np.zeros(5, dtype=np.float64) for name in probe_names}
            val_sample_count = 0
            val_count = {name: 0 for name in probe_names}

            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc='Probe Seed {} Val Epoch {}'.format(seed, epoch + 1)
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1).squeeze()
                    if big_five_data.ndim == 1:
                        big_five_data = big_five_data.unsqueeze(0)

                    audiodata = audiodata.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    outputs = model(
                        fullshot,
                        audiodata
                    )
                    val_sample_count += big_five_data.shape[0]

                    for name in probe_names:
                        sq_err = torch.square(outputs[name] - big_five_data)
                        val_sq_sum[name] += sq_err.sum().item()
                        val_count[name] += big_five_data.numel()
                        val_trait_sq_sum[name] += sq_err.sum(dim=0).detach().cpu().numpy()

            val_mse = {
                name: val_sq_sum[name] / val_count[name]
                for name in probe_names
            }
            val_trait_mse = {name: val_trait_sq_sum[name] / val_sample_count for name in probe_names}

            mean_val_mse = float(
                np.mean([val_mse[name] for name in probe_names])
            )

            # Each representation keeps its own best epoch/result.
            for name in probe_names:
                if val_mse[name] < seed_best[name]['mse']:
                    seed_best[name]['mse'] = float(val_mse[name])
                    seed_best[name]['epoch'] = epoch + 1

            # Patience is based only on the mean validation MSE across probes.
            if mean_val_mse < seed_best_mean_mse:
                seed_best_mean_mse = mean_val_mse
                no_improve_count = 0

                if mean_val_mse < global_best_mean_mse:
                    global_best_mean_mse = mean_val_mse
                    global_best_seed = seed
                    global_best_epoch = epoch + 1

                    torch.save(
                        {
                            'seed': seed,
                            'epoch': epoch + 1,
                            'probe_model': model.probe_heads.state_dict(),
                            'mean_val_mse': mean_val_mse,
                            'per_probe_val_mse': val_mse,
                            'probe_dims': model.probe_dims
                        },
                        probe_best_model_path
                    )
            else:
                no_improve_count += 1

            print(
                '\nSeed {} | Epoch {:03d} | Mean Val MSE={:.6f} | Best Mean Val MSE={:.6f} | Patience={}/{}'.format(
                    seed, epoch + 1, mean_val_mse, seed_best_mean_mse, no_improve_count, patience
                )
            )
            for name in probe_names:
                o_mse,c_mse,e_mse,a_mse,n_mse = val_trait_mse[name]
                print(
                    f'{name:<22} | Train MSE {train_mse[name]:.6f} | '
                    f'Val MSE {val_mse[name]:.6f} | Best MSE {seed_best[name]["mse"]:.6f} | '
                    f'O MSE {o_mse:.6f} | C MSE {c_mse:.6f} | E MSE {e_mse:.6f} | '
                    f'A MSE {a_mse:.6f} | N MSE {n_mse:.6f}'
                )

            if no_improve_count >= patience:
                logger.info(
                    'Probe early stopping: seed={} at epoch {}. '
                    'Best mean val MSE={:.6f}'.format(
                        seed,
                        epoch + 1,
                        seed_best_mean_mse
                    )
                )
                break

            torch.cuda.empty_cache()

        seed_results.append(
            {
                'seed': seed,
                'representations': {
                    name: {
                        'best_epoch': seed_best[name]['epoch'],
                        'best_mse': seed_best[name]['mse']
                    }
                    for name in probe_names
                }
            }
        )

        del model
        del backbone
        del optimizer
        del train_dataloader
        del val_dataloader

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


# ============================================================
# Final multi-seed probe statistics -- direct MSE only
# ============================================================
print('\n' + '=' * 72)
print('FINAL VST LAYERWISE LINEAR PROBE RESULTS -- UDIVA / MSE')
print('=' * 72)
for name in probe_names:
    mses = np.array(
        [result['representations'][name]['best_mse'] for result in seed_results],
        dtype=np.float64
    )
    mse_std = mses.std(ddof=1) if len(mses) > 1 else 0.0
    print(f'{name:<22} MSE {mses.mean():.6f} ± {mse_std:.6f}')
print('-' * 72)
print(
    'Global best mean-probe checkpoint: seed={}, epoch={}, mean MSE={:.6f}'.format(
        global_best_seed, global_best_epoch, global_best_mean_mse
    )
)
print('Saved probe checkpoint: {}'.format(probe_best_model_path))
print('=' * 72)


[2026-09-25 02:22:57,892]::38884681::INFO::================ Probe Seed 1/1: 42 ================


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


[2026-09-25 02:23:01,952]::38884681::INFO::Backbone trainable params: 0
[2026-09-25 02:23:01,953]::38884681::INFO::Probe trainable params: 94175
[2026-09-25 02:23:01,953]::38884681::INFO::Probe dimensions: {'stage1_block1': 96, 'stage1_block2': 96, 'stage1': 192, 'stage2_block1': 192, 'stage2_block2': 192, 'stage2': 384, 'stage3_block1': 384, 'stage3_block2': 384, 'stage3_block3': 384, 'stage3_block4': 384, 'stage3_block5': 384, 'stage3_block6': 384, 'stage3': 768, 'stage4_block1': 768, 'stage4_block2': 768, 'stage4': 1536, 'final_video': 1536, 'final_cross': 1536, 'final_fused': 3072}
Probe Seed 42 Val Epoch 1: 100%|██████████| 191/191 [00:32<00:00,  5.96it/s]



Seed 42 | Epoch 001 | Mean Val MSE=1.353822 | Best Mean Val MSE=1.353822 | Patience=0/10
stage1_block1          | Train MSE 1.118536 | Val MSE 1.116936 | Best MSE 1.116936 | O MSE 1.096812 | C MSE 1.137493 | E MSE 1.095398 | A MSE 1.196226 | N MSE 1.058753
stage1_block2          | Train MSE 0.999216 | Val MSE 1.252646 | Best MSE 1.252646 | O MSE 1.027749 | C MSE 1.246646 | E MSE 1.225116 | A MSE 1.474066 | N MSE 1.289655
stage1                 | Train MSE 0.590202 | Val MSE 1.341829 | Best MSE 1.341829 | O MSE 1.062718 | C MSE 1.580343 | E MSE 1.405816 | A MSE 1.503308 | N MSE 1.156960
stage2_block1          | Train MSE 0.527790 | Val MSE 1.477839 | Best MSE 1.477839 | O MSE 1.345162 | C MSE 1.688022 | E MSE 1.312470 | A MSE 1.589785 | N MSE 1.453757
stage2_block2          | Train MSE 0.755562 | Val MSE 1.384642 | Best MSE 1.384642 | O MSE 1.276396 | C MSE 1.310141 | E MSE 1.446318 | A MSE 1.706738 | N MSE 1.183619
stage2                 | Train MSE 0.477069 | Val MSE 1.380776 | Best 

Probe Seed 42 Val Epoch 2: 100%|██████████| 191/191 [00:32<00:00,  5.86it/s]


...


Seed 42 | Epoch 009 | Mean Val MSE=1.370488 | Best Mean Val MSE=1.353822 | Patience=8/10
stage1_block1          | Train MSE 0.297339 | Val MSE 1.342082 | Best MSE 1.116936 | O MSE 1.334576 | C MSE 1.198163 | E MSE 1.371950 | A MSE 1.307742 | N MSE 1.497977
stage1_block2          | Train MSE 0.289766 | Val MSE 1.414665 | Best MSE 1.252646 | O MSE 1.414209 | C MSE 1.192518 | E MSE 1.430044 | A MSE 1.415673 | N MSE 1.620883
stage1                 | Train MSE 0.065118 | Val MSE 1.363619 | Best MSE 1.341829 | O MSE 1.154422 | C MSE 1.401161 | E MSE 1.458706 | A MSE 1.486654 | N MSE 1.317150
stage2_block1          | Train MSE 0.056657 | Val MSE 1.442041 | Best MSE 1.442041 | O MSE 1.244786 | C MSE 1.554134 | E MSE 1.389839 | A MSE 1.597243 | N MSE 1.424205
stage2_block2          | Train MSE 0.054970 | Val MSE 1.407960 | Best MSE 1.384642 | O MSE 1.235883 | C MSE 1.424599 | E MSE 1.409017 | A MSE 1.588577 | N MSE 1.381725
stage2                 | Train MSE 0.034370 | Val MSE 1.383551 | Best 

Probe Seed 42 Val Epoch 10: 100%|██████████| 191/191 [00:34<00:00,  5.51it/s]



Seed 42 | Epoch 010 | Mean Val MSE=1.376507 | Best Mean Val MSE=1.353822 | Patience=9/10
stage1_block1          | Train MSE 0.283508 | Val MSE 1.349931 | Best MSE 1.116936 | O MSE 1.350979 | C MSE 1.196463 | E MSE 1.391079 | A MSE 1.300135 | N MSE 1.511001
stage1_block2          | Train MSE 0.276278 | Val MSE 1.419728 | Best MSE 1.252646 | O MSE 1.431499 | C MSE 1.183753 | E MSE 1.452409 | A MSE 1.399827 | N MSE 1.631150
stage1                 | Train MSE 0.062186 | Val MSE 1.367854 | Best MSE 1.341829 | O MSE 1.151502 | C MSE 1.413603 | E MSE 1.468949 | A MSE 1.477671 | N MSE 1.327544
stage2_block1          | Train MSE 0.054033 | Val MSE 1.438224 | Best MSE 1.438224 | O MSE 1.238204 | C MSE 1.572868 | E MSE 1.386082 | A MSE 1.578280 | N MSE 1.415683
stage2_block2          | Train MSE 0.051973 | Val MSE 1.405281 | Best MSE 1.384642 | O MSE 1.220632 | C MSE 1.449172 | E MSE 1.402495 | A MSE 1.571197 | N MSE 1.382909
stage2                 | Train MSE 0.033059 | Val MSE 1.397228 | Best 

Probe Seed 42 Val Epoch 11: 100%|██████████| 191/191 [00:21<00:00,  8.98it/s]
[2026-09-25 02:59:09,473]::38884681::INFO::Probe early stopping: seed=42 at epoch 11. Best mean val MSE=1.353822



Seed 42 | Epoch 011 | Mean Val MSE=1.376982 | Best Mean Val MSE=1.353822 | Patience=10/10
stage1_block1          | Train MSE 0.271751 | Val MSE 1.350328 | Best MSE 1.116936 | O MSE 1.361158 | C MSE 1.194884 | E MSE 1.388292 | A MSE 1.294178 | N MSE 1.513128
stage1_block2          | Train MSE 0.264854 | Val MSE 1.419615 | Best MSE 1.252646 | O MSE 1.443518 | C MSE 1.182411 | E MSE 1.452201 | A MSE 1.387755 | N MSE 1.632191
stage1                 | Train MSE 0.059786 | Val MSE 1.371211 | Best MSE 1.341829 | O MSE 1.154700 | C MSE 1.383849 | E MSE 1.479195 | A MSE 1.478628 | N MSE 1.359682
stage2_block1          | Train MSE 0.051799 | Val MSE 1.427245 | Best MSE 1.427245 | O MSE 1.228865 | C MSE 1.521196 | E MSE 1.379228 | A MSE 1.573407 | N MSE 1.433530
stage2_block2          | Train MSE 0.049436 | Val MSE 1.395487 | Best MSE 1.384642 | O MSE 1.212194 | C MSE 1.414720 | E MSE 1.387492 | A MSE 1.562713 | N MSE 1.400315
stage2                 | Train MSE 0.031940 | Val MSE 1.396247 | Best